### TRANSFERIR LOS DATOS A LA DIMENSIÓN TRACK DE LA CAPA GOLD
**IMPORTAMOS LAS LIBRERIAS**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

**EXTRAEMOS LOS DATOS DE LA CAPA SILVER**

In [0]:
gold_silver = spark.table("proyecto_spotify.silver.spotify_tracks")
#display(gold_silver.limit(10))

**FILTRAMOS LOS DATOS DEL TRACK**

In [0]:
dim_track_base = (
    gold_silver
    .select(
        "track_id",
        "track_name",
        "duration_minutes",
        "explicit",
        "album_id"
    )
    .filter(col("track_id").isNotNull())
    .dropDuplicates(["track_id"])
)

**LE GENERAMOS UN CORRELATIVO A CADA REGISTRO**

In [0]:
window_track = Window.orderBy("track_id")

dim_track = (
    dim_track_base
    .withColumn(
        "sk_track",
        row_number().over(window_track)
    )
    .select(
        "sk_track",
        "track_id",
        "track_name",
        "duration_minutes",
        "explicit",
        "album_id"
    )
)

**GUARDAMOS LOS DATOS EN LA TABLA DIM_TRACK DE LA CAPA GOLD**

In [0]:
(
    dim_track
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "proyecto_spotify.gold.dim_track"
    )
)